In [2]:
import sys
import os

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import missingno as msno
import pyarrow
from sqlalchemy import create_engine
from dotenv import load_dotenv

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.figsize'] = (10, 6)

import warnings
warnings.filterwarnings('ignore')

In [3]:
load_dotenv()

USER = os.getenv("LOCAL_DB_USER")
PASSWORD = os.getenv("LOCAL_DB_PASS")
HOST = os.getenv("LOCAL_DB_HOST")
DBNAME = os.getenv("LOCAL_DB_NAME")
PORT = os.getenv("LOCAL_DB_PORT")

DATABASE_URL = f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}"
engine = create_engine(DATABASE_URL)

In [8]:
with engine.connect() as conn:
    df_category = pl.read_database("SELECT * FROM category", conn)
    df_seller = pl.read_database("SELECT * FROM seller", conn)
    df_product = pl.read_database("SELECT * FROM product", conn)
    df_price_offer = pl.read_database("SELECT * FROM price_offer", conn)
    df_review = pl.read_database("SELECT * FROM review", conn)
    df_reviewer = pl.read_database("SELECT * FROM reviewer", conn)
    df_service = pl.read_database("SELECT * FROM service", conn)
    df_coupon = pl.read_database("SELECT * FROM coupon", conn)
    df_offer_coupon = pl.read_database("SELECT * FROM offer_coupon", conn)
    df_offer_service = pl.read_database("SELECT * FROM offer_service", conn)

# print("Tải dữ liệu hoàn tất!")
# print(f"Kích thước bảng Product: {df_product.shape}")
# print(f"Kích thước bảng Review: {df_review.shape}")
df_category.head()

category_id,category_name,parent_category_id,level,category_path,category_url,is_scanned
str,str,str,i64,str,str,bool
"""1789""","""Điện Thoại - Máy Tính Bảng""",null,1,"""Điện Thoại - Máy Tính Bảng""","""https://tiki.vn/dien-thoai-may…",true
"""1686""","""Giày - Dép nam""",null,1,"""Giày - Dép nam""","""https://tiki.vn/giay-dep-nam/c…",true
"""8322""","""Nhà Sách Tiki""",null,1,"""Nhà Sách Tiki""","""https://tiki.vn/nha-sach-tiki/…",true
"""4221""","""Điện Tử - Điện Lạnh""",null,1,"""Điện Tử - Điện Lạnh""","""https://tiki.vn/dien-tu-dien-l…",true
"""8371""","""Đồng hồ và Trang sức""",null,1,"""Đồng hồ và Trang sức""","""https://tiki.vn/dong-ho-va-tra…",true


In [9]:
# Hợp nhất bảng Product và Price Offer để có bức tranh toàn cảnh
df_merged = pd.merge(df_product, df_price_offer, on='product_id', how='inner')
df_clean_price = df_merged.dropna(subset=['current_price', 'discount_percent'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Biểu đồ 1: Phân bố giá bán (Dùng log scale do chênh lệch giá lớn)
sns.histplot(np.log1p(df_clean_price['current_price']), bins=50, kde=True, ax=axes[0], color='dodgerblue')
axes[0].set_title('Phân bố Giá bán hiện tại (Log Scale)')
axes[0].set_xlabel('Log(Giá bán VNĐ)')
axes[0].set_ylabel('Số lượng Sản phẩm')

# Biểu đồ 2: Phân bố mức giảm giá
sns.histplot(df_clean_price['discount_percent'], bins=30, kde=True, ax=axes[1], color='tomato')
axes[1].set_title('Phân bố Mức giảm giá (%)')
axes[1].set_xlabel('Phần trăm giảm (%)')
axes[1].set_ylabel('Số lượng Sản phẩm')

plt.tight_layout()
plt.show()

TypeError: Can only merge Series or DataFrame objects, a <class 'polars.dataframe.frame.DataFrame'> was passed

In [13]:
# Lấy ngẫu nhiên 10,000 mẫu để vẽ Plotly cho nhẹ trình duyệt
df_sample = df_clean_price.sample(n=10000, random_state=42)

# Scatter plot xem mối quan hệ giữa Phần trăm giảm giá và Điểm đánh giá (Review Score)
fig = px.scatter(
    df_sample, 
    x='discount_percent', 
    y='review_score', 
    color='review_score',
    color_continuous_scale='Viridis',
    opacity=0.6,
    title='Mối quan hệ giữa Mức giảm giá và Điểm đánh giá (Mẫu 10k SP)',
    labels={'discount_percent': 'Mức giảm giá (%)', 'review_score': 'Điểm đánh giá trung bình'}
)

fig.update_layout(plot_bgcolor='white')
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# Dùng Polars đếm số lượng từng mức rating siêu tốc
rating_counts_pl = df_review.group_by("rating_score").agg(pl.len().alias("count")).sort("rating_score")
rating_counts = rating_counts_pl.to_pandas()

# Loại bỏ các rating_score không hợp lệ (ví dụ điểm 0 nếu Tiki không hỗ trợ 0 sao)
rating_counts = rating_counts[rating_counts['rating_score'] > 0]

plt.figure(figsize=(8, 5))
sns.barplot(data=rating_counts, x='rating_score', y='count', palette='YlOrRd')
plt.title('Phân bố Số sao đánh giá của Khách hàng (1.3 triệu lượt)', fontsize=16)
plt.xlabel('Số Sao (Rating)')
plt.ylabel('Số lượng Đánh giá')

# Thêm nhãn giá trị trên từng cột
for i, val in enumerate(rating_counts['count']):
    plt.text(i, val + (val*0.02), f'{val:,}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

AttributeError: 'DataFrame' object has no attribute 'group_by'